In [1]:
!pip install gradio

In [2]:
import random
import numpy as np


In [4]:
class Player:
    """
    Represents an independent agent in the game.
    Maintains localized state for memory-based algorithms (like Tit-for-Tat or Naive Bayes).
    """
    def __init__(self, name, strategy_name):
        self.name = name
        self.strategy_name = strategy_name
        self.score = 0
        self.move_history = []
        self.score_history = []  # Tracks cumulative score over time for visualization

    def update_state(self, move, points):
        self.move_history.append(move)
        self.score += points
        self.score_history.append(self.score)

class GameEnvironment:
    """
    Simulates the Prisoner's Dilemma environment, mapping joint actions to utility payoffs.
    """
    def __init__(self, player1, player2, total_rounds):
        self.p1 = player1
        self.p2 = player2
        self.total_rounds = total_rounds

        # Formal Payoff Matrix mapping (Action A, Action B) -> (Utility A, Utility B)
        self.payoff_matrix = {
            ("Cooperate", "Cooperate"): (3, 3),
            ("Betray", "Cooperate"): (5, 0),
            ("Cooperate", "Betray"): (0, 5),
            ("Betray", "Betray"): (1, 1)
        }

    def resolve_round(self, move1, move2):
        """Calculates discrete expected utility based on the payoff matrix."""
        return self.payoff_matrix[(move1, move2)]

In [5]:
import random

class Player:
    def __init__(self, name, strategy_name):
        self.name = name
        self.strategy_name = strategy_name
        self.score = 0
        self.move_history = []
        self.score_history = []

    def update_state(self, move, points):
        self.move_history.append(move)
        self.score += points
        self.score_history.append(self.score)

    def make_move(self, opponent_history):
        """Routes to the correct algorithmic strategy."""
        if self.strategy_name == "Always Cooperate":
            return "Cooperate"

        elif self.strategy_name == "Always Betray":
            return "Betray"

        elif self.strategy_name == "Random":
            return random.choice(["Cooperate", "Betray"])

        elif self.strategy_name == "Tit-for-Tat":
            # Cooperate on the first move, then mimic the opponent's last move
            if not opponent_history:
                return "Cooperate"
            return opponent_history[-1]

        elif self.strategy_name == "Naive Bayes Predictor":
            return self._naive_bayes_decision(opponent_history)

        else:
            return "Cooperate" # Fallback

    def _naive_bayes_decision(self, opp_history):
        """Implements a Naive Bayes classifier to predict opponent's next move."""
        if len(opp_history) < 2:
            return "Cooperate" # Optimistic prior for the first few rounds

        # 1. Calculate Prior Probabilities: P(C) and P(B)
        total_moves = len(opp_history)
        count_c = opp_history.count("Cooperate")
        count_b = total_moves - count_c

        # Laplace smoothing applied to priors
        prior_c = (count_c + 1) / (total_moves + 2)
        prior_b = (count_b + 1) / (total_moves + 2)

        # 2. Calculate Likelihoods: P(O_{t-1} | O_t = C) and P(O_{t-1} | O_t = B)
        last_move = opp_history[-1]

        # Count transitions: How often did 'last_move' precede 'Cooperate' or 'Betray'?
        transitions_to_c = 0
        transitions_to_b = 0

        for i in range(1, total_moves):
            if opp_history[i-1] == last_move:
                if opp_history[i] == "Cooperate":
                    transitions_to_c += 1
                else:
                    transitions_to_b += 1

        # Total times the predicted states actually occurred after any move
        total_c_occurrences = max(1, count_c) # Prevent division by zero mathematically
        total_b_occurrences = max(1, count_b)

        # Laplace smoothed likelihoods
        likelihood_c = (transitions_to_c + 1) / (total_c_occurrences + 2)
        likelihood_b = (transitions_to_b + 1) / (total_b_occurrences + 2)

        # 3. Calculate Proportional Posterior Probabilities
        posterior_c = likelihood_c * prior_c
        posterior_b = likelihood_b * prior_b

        # 4. Decision Rule
        if posterior_c >= posterior_b:
            return "Cooperate"
        else:
            return "Betray"

In [6]:
import matplotlib.pyplot as plt
import pandas as pd

In [7]:
def run_advanced_simulation(strategy_a, strategy_b, total_rounds):
    # 1. Initialize Objects
    p1 = Player("Player A", strategy_a)
    p2 = Player("Player B", strategy_b)
    env = GameEnvironment(p1, p2, total_rounds)

    # 2. Simulation Loop
    log = []
    for r in range(1, total_rounds + 1):
        # Agents observe history and make decisions
        move1 = p1.make_move(p2.move_history)
        move2 = p2.make_move(p1.move_history)

        # Environment resolves utilities
        pts1, pts2 = env.resolve_round(move1, move2)

        # State updates
        p1.update_state(move1, pts1)
        p2.update_state(move2, pts2)

        log.append(f"Round {r}: A ({move1}) vs B ({move2}) -> Score: A(+{pts1}), B(+{pts2})")

    # Formatting final text results
    result_text = "\n".join(log) + f"\n\nFinal Scores:\nPlayer A ({strategy_a}): {p1.score}\nPlayer B ({strategy_b}): {p2.score}"

    if p1.score > p2.score:
        result_text += "\nWinner: Player A Wins" # Requirement from Bonus Features
    elif p2.score > p1.score:
        result_text += "\nWinner: Player B Wins"
    else:
        result_text += "\nResult: Tie"

    # 3. Data Visualization Generation
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Plot A: Cumulative Score Trajectories
    ax1.plot(range(1, total_rounds + 1), p1.score_history, label=f"Player A ({strategy_a})", color='blue', linewidth=2)
    ax1.plot(range(1, total_rounds + 1), p2.score_history, label=f"Player B ({strategy_b})", color='red', linewidth=2)
    ax1.set_title("Cumulative Utility Trajectories")
    ax1.set_xlabel("Rounds")
    ax1.set_ylabel("Total Score")
    ax1.legend()
    ax1.grid(True, linestyle='--', alpha=0.7)

    # Plot B: Cooperation Moving Average (Window = 5)
    p1_coop = [1 if m == "Cooperate" else 0 for m in p1.move_history]
    p2_coop = [1 if m == "Cooperate" else 0 for m in p2.move_history]

    window_size = min(5, total_rounds)
    p1_ma = pd.Series(p1_coop).rolling(window=window_size, min_periods=1).mean()
    p2_ma = pd.Series(p2_coop).rolling(window=window_size, min_periods=1).mean()

    ax2.plot(range(1, total_rounds + 1), p1_ma, label="Player A Cooperation Rate", color='blue', linestyle='-.')
    ax2.plot(range(1, total_rounds + 1), p2_ma, label="Player B Cooperation Rate", color='red', linestyle='-.')
    ax2.set_title(f"Cooperation Frequency (Moving Average, w={window_size})")
    ax2.set_xlabel("Rounds")
    ax2.set_ylabel("Probability of Cooperation")
    ax2.set_ylim(-0.1, 1.1)
    ax2.legend()
    ax2.grid(True, linestyle='--', alpha=0.7)

    plt.tight_layout()

    return result_text, fig

In [ ]:
import gradio as gr

# Define the extended strategic space
strategy_space = [
    "Always Cooperate",
    "Always Betray",
    "Random",
    "Tit-for-Tat",
    "Naive Bayes Predictor"
]

# Construct the Interactive Interface
app = gr.Interface(
    fn=run_advanced_simulation,
    inputs=[
        gr.Dropdown(
            choices=strategy_space,
            label="Player A Strategy",
            value="Naive Bayes Predictor"
        ),
        gr.Dropdown(
            choices=strategy_space,
            label="Player B Strategy",
            value="Tit-for-Tat"
        ),
        gr.Slider(
            minimum=5,
            maximum=100,
            step=1,
            value=20,
            label="Rounds (Time Horizon: $T$)"
        )
    ],
    outputs=[
        gr.Textbox(label="Simulation Log & Final Utilities", lines=12),
        gr.Plot(label="Strategic Trajectories & Behavioral Convergence")
    ],
    title="Advanced Game Theory Simulator: Prisoner's Dilemma",
    description="An interactive simulation engine evaluating deterministic algorithms and bounded rationality (Markovian Naive Bayes) converging toward Nash Equilibria."
)

# Initialize the server
app.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://49038f734873696e97.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
